# 第15回 機械学習・深層学習による予測

このノートブックは、スライドと同じ順に並んでいます。
コードはすべて完成しているので、上から順に実行して、結果を読んでください。
この回だけ、RではなくPythonを使います。

- 0節：セットアップ（授業の最初に実行する）
- A節：理論をシミュレーションで確かめる
- B節：実データを解析する
- C節：設定を変えて回し直す

今日の問い：予測の問題を、機械学習が解ける形に直すには、何を特徴量にすればよいか。

## 0. セットアップ

授業が始まったら、まずこのセルを実行してください。
ライブラリのインストールに少し時間がかかります。終わるまで、次のセルは動きません。

In [ ]:
# Python の版を確認
import sys
print(sys.version)

# この回で使うライブラリを、版を固定して入れる
!pip install -q mlforecast==1.1.0 lightgbm==4.7.0 chronos-forecasting==2.3.2

# 入った版を確認
from importlib.metadata import version
for p in ["numpy", "pandas", "scikit-learn", "matplotlib", "torch",
          "lightgbm", "mlforecast", "chronos-forecasting"]:
    print(p, version(p))

## A. 理論をシミュレーションで確かめる

In [ ]:
import os
import warnings
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"   # モデルの配布元に、ログインの情報を送らない（B-7。ログインは要らない）
os.environ["HF_HUB_VERBOSITY"] = "error"            # 「ログインしていない」という案内を出さない
warnings.filterwarnings("ignore", module="huggingface_hub")
import torch                                   # 深層学習のライブラリ（B-7で使う）
from chronos import BaseChronosPipeline        # 事前学習済みモデル（B-7で使う）
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score
from mlforecast import MLForecast
from mlforecast.target_transforms import Differences
plt.rcParams["figure.figsize"] = (8, 4.5)

誤差の測り方は、第2回と同じです。二乗平均平方根誤差（RMSE）と平均絶対誤差（MAE）を使います。
ベンチマーク（平均法、ナイーブ法、季節ナイーブ法）も、第2回と同じものを自分で書いておきます。

In [ ]:
# 誤差（第2回と同じ）
def rmse(estimate, truth):
    return float(np.sqrt(np.mean((np.asarray(estimate) - np.asarray(truth)) ** 2)))

def mae(estimate, truth):
    return float(np.mean(np.abs(np.asarray(estimate) - np.asarray(truth))))

# 予測の名前と値の組から、誤差の表を作る
def accuracy(preds, truth):
    return pd.DataFrame({"RMSE": {k: rmse(v, truth) for k, v in preds.items()},
                         "MAE": {k: mae(v, truth) for k, v in preds.items()}})

# ベンチマーク（第2回と同じ）：y は訓練データ、h は予測する個数、m は季節の周期
def benchmarks(y, h, m):
    return {"mean": np.full(h, y.mean()),      # 平均法
            "naive": np.full(h, y[-1]),        # ナイーブ法
            "snaive": np.resize(y[-m:], h)}    # 季節ナイーブ法

### A-1. ラグ特徴量：予測を、表の形の回帰に直す

機械学習のモデルは、「1行が1つの例、列が特徴量（feature）」という表を受け取ります。
時系列は1本の列なので、そのままでは渡せません。

そこで、系列を1つずつ下にずらした列を横に並べます。ずらした列を、ラグ特徴量（lag feature）と呼びます。

まず、AR(2)モデルから系列を作ります（第3回）。

$$
y_t = 0.5\,y_{t-1} + 0.3\,y_{t-2} + w_t
$$

In [ ]:
rng = np.random.default_rng(1)
n = 1100
w = rng.normal(size=n)
y = np.zeros(n)
for t in range(2, n):
    y[t] = 0.5 * y[t - 1] + 0.3 * y[t - 2] + w[t]

plt.plot(y[-200:], color="black", linewidth=0.8)
plt.xlabel("Time (last 200 points)")
plt.ylabel("Value (AR(2) series)")
plt.show()

In [ ]:
# 系列から、ラグ特徴量の表を作る（ずらして値がなくなった行は捨てる）
def make_lags(y, lags):
    tab = pd.DataFrame({"y": y})
    for k in lags:
        tab[f"lag{k}"] = tab["y"].shift(k)
    return tab.dropna()

In [ ]:
tab = make_lags(y, [1, 2])
print(tab.shape)
tab.head()

- 1行が、1つの時点です。`y`が当てたい値、`lag1`が1時点前の値、`lag2`が2時点前の値です。
- 最初の2行は、ずらすと値がないので捨てています。

この表に、線形回帰を当てはめます。最後の100行をテストデータにします。

In [ ]:
X = ["lag1", "lag2"]
train, test = tab.iloc[:-100], tab.iloc[-100:]
fit_lr = LinearRegression().fit(train[X], train["y"])
coef_ar2 = pd.Series(fit_lr.coef_, index=X).round(3)
coef_ar2

係数は、系列を作るときに使った0.5と0.3に近い値になります。
ラグ特徴量の線形回帰は、第3回の「ARモデルを最小二乗法で推定する」と同じ計算です。

### A-2. 勾配ブースティング：線形でない関係を学ぶ

次の系列は、1時点前の値との関係が直線ではありません。

$$
y_t = 2.5 \sin(y_{t-1}) + w_t
$$

In [ ]:
rng = np.random.default_rng(2)
w = rng.normal(size=n)
z = np.zeros(n)
for t in range(1, n):
    z[t] = 2.5 * np.sin(z[t - 1]) + w[t]

tab_z = make_lags(z, [1])
plt.scatter(tab_z["lag1"], tab_z["y"], s=5, color="grey")
plt.xlabel("Value at time t-1 (lag1)")
plt.ylabel("Value at time t (y)")
plt.show()

勾配ブースティング（gradient boosting）は、小さな決定木（decision tree）を1本ずつ足していく方法です。
新しい木は、それまでの木の予測が外した分（残差）を当てるように作ります。
ここでは LightGBM というライブラリを使います。設定は、この関数にまとめておきます。

In [ ]:
# 勾配ブースティングのモデル（木の数 n_trees、1本の木の葉の数は8まで）
def make_lgb(n_trees=200):
    return lgb.LGBMRegressor(n_estimators=n_trees, learning_rate=0.05, num_leaves=8,
                             importance_type="gain", random_state=1, n_jobs=1, verbose=-1)

木の数を1本、10本、200本と増やして、当てはめた関数を描きます。

In [ ]:
train_z, test_z = tab_z.iloc[:-100], tab_z.iloc[-100:]
grid = pd.DataFrame({"lag1": np.linspace(-5, 5, 400)})

plt.scatter(train_z["lag1"], train_z["y"], s=5, color="lightgrey")
for n_trees in [1, 10, 200]:
    fit = make_lgb(n_trees).fit(train_z[["lag1"]], train_z["y"])
    plt.plot(grid["lag1"], fit.predict(grid), label=f"{n_trees} trees")
plt.plot(grid["lag1"], 2.5 * np.sin(grid["lag1"]), color="black", linestyle="--", label="true function")
plt.xlabel("Value at time t-1 (lag1)")
plt.ylabel("Value at time t (y)")
plt.legend()
plt.show()

- 木は、特徴量の範囲を区切り、区切りごとに一定の値を返します。そのため、当てはめた関数は階段の形になります。
- 木を足すほど、階段が細かくなり、本当の関数（破線）に近づきます。

2本の系列（AR(2)と、線形でない系列）で、1時点先の予測の誤差を比べます。

In [ ]:
def onestep(tab, X):
    train, test = tab.iloc[:-100], tab.iloc[-100:]
    preds = {"mean": np.full(100, train["y"].mean()),
             "linear": LinearRegression().fit(train[X], train["y"]).predict(test[X]),
             "lgb": make_lgb().fit(train[X], train["y"]).predict(test[X])}
    return accuracy(preds, test["y"])["RMSE"]

res_onestep = pd.DataFrame({"AR(2)": onestep(tab, ["lag1", "lag2"]),
                            "nonlinear": onestep(tab_z, ["lag1"])})
res_onestep.round(2)

- AR(2)の系列：線形回帰のほうが、少し良い。関係が直線なら、直線のモデルで足ります。
- 線形でない系列：線形回帰は平均法とあまり変わらず、勾配ブースティングは、ノイズの標準偏差（1）に近い誤差になります。

### A-3. 木は、見たことのない範囲を当てられない

右上がりの系列（ドリフトつきランダムウォーク、第4回）を作り、最後の50個を予測します。

ここからは、mlforecast というライブラリを使います。mlforecast は、ラグ特徴量の表を作り、モデルを当てはめ、先の値を1時点ずつ順に予測します。
2時点先を予測するときは、1時点先の予測値を`lag1`に入れます。これを、再帰的な予測（recursive forecast）と呼びます。

mlforecast に渡す表は、列の名前が決まっています。`unique_id`が系列の名前、`ds`が時点、`y`が値です。

In [ ]:
rng = np.random.default_rng(3)
u = np.cumsum(0.2 + 0.5 * rng.normal(size=500))
df_u = pd.DataFrame({"unique_id": "u", "ds": np.arange(500), "y": u})
h = 50
train_u, test_u = df_u.iloc[:-h], df_u.iloc[-h:]

mf = MLForecast(models={"linear": LinearRegression(), "lgb": make_lgb()}, freq=1, lags=[1, 2])
fc_raw = mf.fit(train_u).predict(h)
fc_raw.tail(3)

In [ ]:
# 系列の終わりのほうと、予測を重ねて描く
def plot_forecast(df, fc, last, ylabel):
    past = df.iloc[-last:]
    plt.plot(past["ds"], past["y"], color="black", linewidth=0.8, label="actual")
    for name in fc.columns.drop(["unique_id", "ds"]):
        plt.plot(fc["ds"], fc[name], label=name)
    plt.xlabel("Time")
    plt.ylabel(ylabel)
    plt.legend()

In [ ]:
plot_forecast(df_u, fc_raw, 200, "Value (random walk with drift)")
plt.axhline(train_u["y"].max(), color="grey", linestyle=":")   # 訓練データの最大値
plt.show()

勾配ブースティングの予測は、訓練データの最大値（点線）を超えません。
木が返す値は、訓練データの`y`の平均を組み合わせたものだからです。

第4回と同じように、差分をとってから学習させます。予測は、差分を足し上げて、もとの大きさに戻します。

In [ ]:
mf_d = MLForecast(models={"linear": LinearRegression(), "lgb": make_lgb()}, freq=1, lags=[1, 2],
                  target_transforms=[Differences([1])])
fc_diff = mf_d.fit(train_u).predict(h)
plot_forecast(df_u, fc_diff, 200, "Value (random walk with drift)")
plt.show()

res_trend = pd.DataFrame({"raw": accuracy({"linear": fc_raw["linear"], "lgb": fc_raw["lgb"]}, test_u["y"])["RMSE"],
                          "differenced": accuracy({"linear": fc_diff["linear"], "lgb": fc_diff["lgb"]}, test_u["y"])["RMSE"]})
res_trend.round(2)

### A-4. ランダムな交差検証を、時系列に使うと

機械学習のライブラリの交差検証は、既定では、行をランダムに分けます（または、行の順を無視して分けます）。
第2回で見たとおり、これは iid を仮定した評価です。

3本の系列で、次の3つを比べます。モデルは、ラグ特徴量（`lag1`、`lag2`）の勾配ブースティングです。

- `kfold`：行をランダムに5つに分ける交差検証で見積もった誤差
- `tscv`：時間の順を守る交差検証（`TimeSeriesSplit`）で見積もった誤差
- `future`：そのあとの500時点を、実際に（1時点先ずつ）予測したときの誤差

In [ ]:
def cv_rmse(tab, cv):
    score = cross_val_score(make_lgb(), tab[["lag1", "lag2"]], tab["y"], cv=cv,
                            scoring="neg_root_mean_squared_error")
    return -score.mean()

def compare_cv(series):
    tab = make_lags(series, [1, 2])
    past, future = tab.iloc[:-500], tab.iloc[-500:]
    fit = make_lgb().fit(past[["lag1", "lag2"]], past["y"])
    return {"kfold": cv_rmse(past, KFold(5, shuffle=True, random_state=1)),
            "tscv": cv_rmse(past, TimeSeriesSplit(5)),
            "future": rmse(fit.predict(future[["lag1", "lag2"]]), future["y"])}

rng = np.random.default_rng(2)
w = rng.normal(size=1500)
ar1 = np.zeros(1500)
for t in range(1, 1500):
    ar1[t] = 0.8 * ar1[t - 1] + w[t]

res_cv = pd.DataFrame({"white noise": compare_cv(w),
                       "AR(1)": compare_cv(ar1),
                       "random walk with drift": compare_cv(np.cumsum(0.2 + w))}).T
res_cv.round(2)

- ホワイトノイズと AR(1)：3つの数は、ほぼ同じです。
- ドリフトつきランダムウォーク：ランダムに分ける交差検証は、誤差をずっと小さく見積もります。隠した行と同じ高さの行が、訓練データに入っているからです。実際の予測では、見たことのない高さの値が来ます。

### A-5. フーリエ項：周期の中の位置を、特徴量にする

1年の周期をもつ、毎日の系列を作ります（3年分）。最後の180日を予測します。

In [ ]:
# フーリエ項：周期 period の正弦波と余弦波を K 組。t は時点の番号
def fourier_terms(t, period, K):
    cols = {}
    for k in range(1, K + 1):
        cols[f"sin{k}"] = np.sin(2 * np.pi * k * t / period)
        cols[f"cos{k}"] = np.cos(2 * np.pi * k * t / period)
    return pd.DataFrame(cols)

In [ ]:
rng = np.random.default_rng(4)
day = np.arange(3 * 365)
season = 10 * np.cos(2 * np.pi * day / 365) + 5 * np.cos(4 * np.pi * day / 365 + 1)
s = 50 + season + rng.normal(scale=3, size=len(day))
h = 180

plt.plot(day, s, color="grey", linewidth=0.6)
plt.axvline(len(day) - h, color="black", linestyle=":")   # ここから先を予測する
plt.xlabel("Day")
plt.ylabel("Value (series with a yearly cycle)")
plt.show()

第7回で、周期のある動きは、正弦波と余弦波の和で表せることを見ました。
その正弦波と余弦波を、特徴量の列として表に足します。これをフーリエ項（Fourier terms）と呼びます。

フーリエ項の組の数 $K$ を変えて、線形回帰で予測します。

In [ ]:
res_K = {}
for K in range(0, 5):
    if K == 0:
        pred = np.full(h, s[:-h].mean())
    else:
        F = fourier_terms(day, 365, K)
        pred = LinearRegression().fit(F.iloc[:-h], s[:-h]).predict(F.iloc[-h:])
    res_K[f"K={K}"] = rmse(pred, s[-h:])
res_K["snaive (365 days)"] = rmse(s[-h - 365:-365], s[-h:])
pd.Series(res_K).round(2)

- $K = 2$ で、誤差はノイズの標準偏差（3）に近くなります。この系列は、波を2つ足して作ったからです。
- 季節ナイーブ法（365日前の値）は、ノイズを2回分含むので、誤差が大きくなります。

ラグ特徴量だけのモデルと、フーリエ項を足したモデルを、mlforecast で比べます。

In [ ]:
df_s = pd.DataFrame({"unique_id": "s", "ds": day, "y": s})
df_sF = pd.concat([df_s, fourier_terms(day, 365, 2)], axis=1)

fc_lags = MLForecast(models={"lgb": make_lgb()}, freq=1, lags=[1, 7]).fit(df_s.iloc[:-h]).predict(h)
mf_F = MLForecast(models={"lgb": make_lgb()}, freq=1, lags=[1, 7])
mf_F.fit(df_sF.iloc[:-h], static_features=[])
fc_four = mf_F.predict(h, X_df=df_sF.iloc[-h:].drop(columns="y"))

fc_both = fc_lags.rename(columns={"lgb": "lags only"}).assign(**{"lags + Fourier": fc_four["lgb"]})
plot_forecast(df_s, fc_both, 500, "Value (series with a yearly cycle)")
plt.show()
res_fourier = accuracy({"lags only": fc_both["lags only"], "lags + Fourier": fc_both["lags + Fourier"]}, s[-h:])
res_fourier.round(2)

ラグ特徴量だけでは、「いまが1年のどこか」が分かりません。先の時点の予測は、平らになります。
フーリエ項は、先の時点でも値が分かっている特徴量です。

### A-6. グローバルモデル：たくさんの系列を、1つのモデルで学ぶ

短い系列が200本あるとします。モデルの作り方は、2通りあります。

- ローカルモデル（local model）：系列ごとに、別々のモデルを当てはめる。
- グローバルモデル（global model）：すべての系列の行を1つの表に積み、1つのモデルを当てはめる。

200本の AR(1) の系列を作り、最後の1時点を、ラグ特徴量の線形回帰で予測します。
系列の長さ（20か200）と、係数 $\phi$ が全系列で同じかどうかを変えます。

In [ ]:
# AR(1) の系列を N 本作る（phis は系列ごとの係数、T は長さ）
def sim_panel(phis, T, seed):
    rng = np.random.default_rng(seed)
    rows = []
    for i, phi in enumerate(phis):
        x = np.zeros(T + 20)
        for t in range(1, T + 20):
            x[t] = phi * x[t - 1] + rng.normal()
        rows.append(pd.DataFrame({"unique_id": i, "ds": np.arange(T), "y": x[20:]}))
    return pd.concat(rows, ignore_index=True)

# 各系列の最後の1時点を、ローカルモデルとグローバルモデルで予測して、RMSE を比べる
def compare_global_local(panel):
    tab = panel.assign(lag1=panel.groupby("unique_id")["y"].shift(1),
                       lag2=panel.groupby("unique_id")["y"].shift(2)).dropna()
    last = tab["ds"] == tab["ds"].max()
    train, test = tab[~last], tab[last]
    X = ["lag1", "lag2"]
    pred_global = LinearRegression().fit(train[X], train["y"]).predict(test[X])
    pred_local = [LinearRegression().fit(d[X], d["y"]).predict(test.loc[test["unique_id"] == i, X])[0]
                  for i, d in train.groupby("unique_id")]
    return {"local": rmse(pred_local, test["y"]), "global": rmse(pred_global, test["y"])}

In [ ]:
N = 200
same = np.full(N, 0.7)                                                # 全系列で同じ係数
mixed = np.where(np.random.default_rng(6).random(N) < 0.5, 0.8, -0.8)   # 系列によって係数が違う
res_global = pd.DataFrame({
    "same phi, T=20": compare_global_local(sim_panel(same, 20, 5)),
    "same phi, T=200": compare_global_local(sim_panel(same, 200, 5)),
    "mixed phi, T=20": compare_global_local(sim_panel(mixed, 20, 5)),
    "mixed phi, T=200": compare_global_local(sim_panel(mixed, 200, 5))}).T
res_global.round(3)

- 係数が同じで、系列が短いとき：グローバルモデルのほうが良い。1本あたり18行しかなくても、200本を合わせれば3600行になります。
- 系列が長いとき：ローカルモデルも、係数を十分に推定できます。差は小さくなります。
- 係数が系列によって違うとき：グローバルモデルは「すべての系列が同じ式に従う」と仮定しているので、その仮定が外れます。

## B. 実データを解析する

2つのデータを使います。どちらも、Rの`tsibbledata`パッケージのデータをCSVに書き出したものです。

- `vic_elec`：オーストラリアのビクトリア州の電力需要。第2回と同じく、1日ごとにまとめてあります。
- `tourism`：オーストラリアの国内旅行の宿泊数。四半期ごと、地域と目的の組み合わせで304系列あります。

B節は、A節で作った関数（`make_lgb`など）を使います。A節を実行してから進んでください。

### B-0. 設定

このセルの値を変えると、B節の解析がすべて変わります。まずはこのまま実行してください。

In [ ]:
# ここを変えると、以下の解析がすべて変わる
series    = "Demand"    # 系列："Demand"（1日の電力需要）か "Temperature"（1日の最高気温）
year_from = 2012        # 訓練データの始まりの年：2012, 2013, 2014 のどれか
h         = 28          # 予測する日数：7, 14, 28 のどれか
lags      = [1, 7]      # ラグ特徴量：[1]、[1, 7]、[7, 14]、[1, 7, 14] のどれか
K         = 2           # 1年周期のフーリエ項の組の数：0〜3
calendar  = True        # 曜日と祝日を特徴量に入れるか：True か False
purpose   = "Holiday"   # 旅行の目的（B-8）："Business" "Holiday" "Other" "Visiting" のどれか

### B-1. データを読んで、確かめる

In [ ]:
DATA = "https://raw.githubusercontent.com/k0heiun0/timeseries_preview/main/data/"
elec = pd.read_csv(DATA + "vic_elec_daily.csv", parse_dates=["Date"])
tour = pd.read_csv(DATA + "tourism.csv", parse_dates=["Quarter"])
print(elec.head(3))
print(elec["Date"].min(), elec["Date"].max(), len(elec))
print("missing:", int(elec.isna().sum().sum()), " gaps:", int((elec["Date"].diff().dt.days[1:] != 1).sum()))
print(tour.head(3))
print(tour["Quarter"].min(), tour["Quarter"].max(), len(tour))
print("series:", len(tour.groupby(["Region", "Purpose"])), " missing:", int(tour.isna().sum().sum()))

- `elec`：1行は1日です。`Demand`が1日の電力需要（GWh）、`Temperature`が1日の最高気温（℃）、`Holiday`が祝日かどうか（1か0）です。2012年1月1日から2014年12月31日までの1096行で、欠測も日付の抜けもありません。
- `tour`：1行は、1つの地域・1つの目的・1つの四半期です。`Trips`が宿泊をともなう旅行の数（千回）です。1998年の第1四半期から2017年の第4四半期までの80期、304系列で、欠測はありません。

### B-2. 前処理：特徴量の表を作る

期間を絞り、mlforecast に渡す形（`unique_id`、`ds`、`y`）にします。
そのあと、先の日でも値が分かっている特徴量を足します。

- フーリエ項（`sin1`、`cos1`、…）：1年（365.25日）の周期の中の位置
- 曜日（`dow1`〜`dow6`）：火曜から日曜までを、それぞれ1か0で表す（月曜はすべて0）
- 祝日（`holiday`）

In [ ]:
d = elec[elec["Date"].dt.year >= year_from].reset_index(drop=True)
df = pd.DataFrame({"unique_id": series, "ds": d["Date"], "y": d[series]})
t = (d["Date"] - pd.Timestamp("2012-01-01")).dt.days        # 2012年1月1日からの日数
df = pd.concat([df, fourier_terms(t, 365.25, K)], axis=1)
if calendar:
    df["holiday"] = d["Holiday"]
    for j in range(1, 7):
        df[f"dow{j}"] = (d["Date"].dt.dayofweek == j).astype(int)

y_lab = "Daily demand (GWh)" if series == "Demand" else "Daily max temperature (deg C)"
n = len(df)
train, test = df.iloc[:n - h], df.iloc[n - h:]     # 最後の h 日がテストデータ
print(n, len(train), len(test))
df.tail(3)

In [ ]:
plt.plot(train["ds"], train["y"], color="black", linewidth=0.6, label="training")
plt.plot(test["ds"], test["y"], color="red", linewidth=0.8, label="test")
plt.xlabel("Date")
plt.ylabel(y_lab)
plt.legend()
plt.show()

### B-3. ベンチマーク

第2回と同じ3つのベンチマークで、最後の`h`日を予測します。
系列が`"Demand"`、`h`が28のとき、テストデータは第2回と同じ期間です。ナイーブ法と季節ナイーブ法の誤差は、第2回の値と一致します。

In [ ]:
preds = benchmarks(train["y"].to_numpy(), h, 7)
accuracy(preds, test["y"]).round(2)

### B-4. モデルを当てはめて、予測する

mlforecast が作る表を見ます。ラグ特徴量が、右端に足されています。

In [ ]:
mf = MLForecast(models={"linear": LinearRegression(), "lgb": make_lgb()}, freq="D", lags=lags)
mf.preprocess(train, static_features=[]).head(3)

同じ特徴量で、線形回帰と勾配ブースティングを当てはめ、`h`日先まで予測します。

In [ ]:
mf.fit(train, static_features=[], fitted=True)
future_X = test.drop(columns="y")                  # テスト期間の、先に分かっている特徴量
fc = mf.predict(h, X_df=future_X if future_X.shape[1] > 2 else None)
preds["linear"] = fc["linear"].to_numpy()
preds["lgb"] = fc["lgb"].to_numpy()

plot_forecast(df, fc, 3 * h, y_lab)
plt.plot(test["ds"], preds["snaive"], linestyle="--", color="grey", label="snaive")
plt.legend()
plt.show()

In [ ]:
acc = accuracy(preds, test["y"])
acc.round(2)

### B-5. 診断

勾配ブースティングが、どの特徴量をよく使ったかを見ます。
重要度（importance）は、その特徴量で区切ったことで、誤差がどれだけ減ったかの合計です。

In [ ]:
importance = pd.Series(mf.models_["lgb"].feature_importances_,
                       index=mf.ts.features_order_).sort_values()
(importance / importance.sum()).plot.barh(color="grey")
plt.xlabel("Share of importance (gain)")
plt.show()

訓練データでの1時点先の予測の残差について、標本ACFを描きます（第1回）。

In [ ]:
# 標本ACFを棒で描く。破線は ±1.96/√n
def plot_acf(x, lag_max, ylabel):
    x = np.asarray(x) - np.mean(x)
    acf = np.array([np.sum(x[k:] * x[:len(x) - k]) for k in range(1, lag_max + 1)]) / np.sum(x * x)
    plt.bar(np.arange(1, lag_max + 1), acf, width=0.3, color="black")
    for b in (-1.96, 1.96):
        plt.axhline(b / np.sqrt(len(x)), color="blue", linestyle="--")
    plt.axhline(0, color="black", linewidth=0.5)
    plt.xlabel("Lag h (day)")
    plt.ylabel(ylabel)
    return acf

In [ ]:
fitted = mf.forecast_fitted_values()
resid = fitted["y"] - fitted["lgb"]
resid_acf = plot_acf(resid, 28, "Sample ACF of residuals (lgb)")
plt.show()

### B-6. 時系列交差検証

第2回と同じ形の時系列交差検証をします。訓練データを`h`日ずつ伸ばしながら、6回評価します。6回目が、B-4の評価です。

In [ ]:
# ベンチマークの時系列交差検証（mlforecast の cross_validation と同じ区切り方）
def benchmarks_cv(y, h, m, n_windows):
    out = []
    for i in range(n_windows, 0, -1):
        end = len(y) - i * h                       # 訓練データの終わり
        out.append(pd.DataFrame(benchmarks(y[:end], h, m)).assign(y=y[end:end + h]))
    return pd.concat(out, ignore_index=True)

In [ ]:
cv = mf.cross_validation(df, n_windows=6, h=h, step_size=h, static_features=[])
print(cv.groupby("cutoff")["ds"].agg(["min", "max"]))
cv_bench = benchmarks_cv(df["y"].to_numpy(), h, 7, 6)
acc_cv = accuracy({"mean": cv_bench["mean"], "naive": cv_bench["naive"], "snaive": cv_bench["snaive"],
                   "linear": cv["linear"], "lgb": cv["lgb"]}, cv["y"])
acc_cv.round(2)

### B-7. 事前学習済みモデルで、学習せずに予測する

たくさんの時系列であらかじめ学習してあるモデルを、基盤モデル（foundation model）と呼びます。
ここでは、Chronos-Bolt（small）を使います。モデルの重み（約190MB）をダウンロードします。ログインは要りません。

ここでは、モデルの学習（係数の推定）を一切しません。出来上がっているモデルに、訓練データの`y`の列だけを渡して、続きを予測させます。特徴量も渡しません。
こういう使い方を、ゼロショット予測（zero-shot forecasting）と呼びます。

In [ ]:
pipe = BaseChronosPipeline.from_pretrained("amazon/chronos-bolt-small", device_map="cpu")

In [ ]:
context = torch.tensor(train["y"].to_numpy(), dtype=torch.float32)
q, _ = pipe.predict_quantiles(context, prediction_length=h, quantile_levels=[0.1, 0.5, 0.9])
q = q[0].numpy()                                   # 列は、10%点、中央値、90%点
preds["chronos"] = q[:, 1]

past = df.iloc[-3 * h:]
plt.plot(past["ds"], past["y"], color="black", linewidth=0.8, label="actual")
plt.plot(test["ds"], q[:, 1], color="tab:green", label="chronos (median)")
plt.fill_between(test["ds"], q[:, 0], q[:, 2], color="tab:green", alpha=0.2, label="10%-90%")
plt.xlabel("Time")
plt.ylabel(y_lab)
plt.legend()
plt.show()

acc = accuracy(preds, test["y"])
acc.round(2)

### B-8. たくさんの系列：グローバルモデル

`tourism`から、設定で選んだ目的の76系列（76地域）を取り出します。最後の8期（2年）をテストデータにします。

In [ ]:
tr = tour[tour["Purpose"] == purpose]
panel = pd.DataFrame({"unique_id": tr["Region"], "ds": tr["Quarter"], "y": tr["Trips"]})
h_q = 8
cut = np.sort(panel["ds"].unique())[-h_q]          # テストデータの最初の期
train_p, test_p = panel[panel["ds"] < cut], panel[panel["ds"] >= cut]
print(panel["unique_id"].nunique(), len(train_p), len(test_p))

for name in panel["unique_id"].unique()[:4]:
    one = panel[panel["unique_id"] == name]
    plt.plot(one["ds"], one["y"], linewidth=0.8, label=name)
plt.xlabel("Quarter")
plt.ylabel("Overnight trips (thousands)")
plt.legend()
plt.show()

3つの方法で予測します。ラグ特徴量は、1〜4期前と8期前です。

- `snaive`：季節ナイーブ法（4期前の値）。系列ごと。
- `local`：系列ごとに、別々の線形回帰（ローカルモデル）。
- `global linear`、`global lgb`：76系列を1つの表に積んで、1つのモデル（グローバルモデル）。

In [ ]:
lags_q = [1, 2, 3, 4, 8]
fc_g = MLForecast(models={"linear": LinearRegression(), "lgb": make_lgb()},
                  freq="QS", lags=lags_q).fit(train_p).predict(h_q)
fc_l = pd.concat([MLForecast(models={"linear": LinearRegression()}, freq="QS", lags=lags_q).fit(one).predict(h_q)
                  for _, one in train_p.groupby("unique_id")])
res = (test_p.merge(fc_g, on=["unique_id", "ds"])
             .merge(fc_l.rename(columns={"linear": "local"}), on=["unique_id", "ds"])
             .sort_values(["unique_id", "ds"]))
res["snaive"] = np.concatenate([benchmarks(one["y"].to_numpy(), h_q, 4)["snaive"]
                                for _, one in train_p.groupby("unique_id")])

# 系列ごとの RMSE
methods = {"snaive": "snaive", "local": "local", "global linear": "linear", "global lgb": "lgb"}
by_series = pd.DataFrame({name: res.groupby("unique_id").apply(lambda g: rmse(g[col], g["y"]), include_groups=False)
                          for name, col in methods.items()})
acc_tour = pd.DataFrame({"mean RMSE": by_series.mean(),
                         "wins vs snaive": by_series.lt(by_series["snaive"], axis=0).sum()})
acc_tour.round(2)

- `mean RMSE`は、系列ごとのRMSEを、76系列で平均したものです。
- `wins vs snaive`は、76系列のうち、季節ナイーブ法よりRMSEが小さかった系列の数です。

In [ ]:
plt.scatter(by_series["local"], by_series["global linear"], s=12, color="black")
lim = by_series[["local", "global linear"]].to_numpy().max()
plt.plot([0, lim], [0, lim], color="grey", linestyle="--")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("RMSE of local model (thousands of trips)")
plt.ylabel("RMSE of global model (thousands of trips)")
plt.show()

1つの点が1つの系列です。破線より下にある系列では、グローバルモデルのほうが誤差が小さくなっています。

読み方：

- B-4：機械学習のモデルが、季節ナイーブ法に勝つとは限りません。勝ったときは、どの特徴量が効いたかをB-5で確かめます。
- B-6：1回の評価（B-4）と6回の評価で、順位が変わることがあります。
- B-7：基盤モデルは、祝日を知りません。それでも当たるか、どこで外れるかを、図で見ます。
- B-8：グローバルモデルは「どの地域も、同じ式に従う」と仮定しています。

## C. 設定を変えて回し直す

B-0の設定セルを書き替えて、B節を上からすべて実行し直してください
（Colabでは、設定セルを選んで、メニューの［ランタイム］→［以降のセルを実行］）。

変えられる値：

| 変数 | 値 |
| --- | --- |
| `series` | `"Demand"` `"Temperature"` |
| `year_from` | 2012、2013、2014 |
| `h` | 7、14、28 |
| `lags` | `[1]` `[1, 7]` `[7, 14]` `[1, 7, 14]` |
| `K` | 0、1、2、3 |
| `calendar` | `True` `False` |
| `purpose` | `"Business"` `"Holiday"` `"Other"` `"Visiting"` |

読むときの観点：

1. `calendar`を`False`にすると、線形回帰と勾配ブースティングの誤差はどう変わるか。重要度の図では、何が上に来るか。
2. `year_from`を2014にすると（訓練データが1年分だけ）、フーリエ項（`K`が1以上）は役に立つか。`K`を0にするとどうなるか。
3. 系列を`"Temperature"`にすると、どの方法がナイーブ法に勝つか。
4. `lags`を`[7, 14]`にすると、再帰的な予測はどう変わるか（`h`が7以下なら、予測値をラグに入れる必要がなくなる）。
5. `purpose`を変えると、グローバルモデルとローカルモデルの差はどう変わるか。

正解は1つではありません。自分の設定で出た結果を、最初の設定の結果と比べてください。